A Parallel Workflow using LLM calls to generate a feedback and score for a given content. 
The workflow is designed to take text input, evaluate the text content to provide feedback and a score.

In [ ]:
# from langgraph.graph import StateGraph, START, END
# from typing import TypedDict, Annotated
# # from load_dotenv import load_dotenv
# import operator

# # load_dotenv()

In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
# from load_dotenv import load_dotenv
import operator

# load_dotenv()

In [ ]:
# Define models and prompts for the UPSC essay workflow

# Create a OpenAI model instance
from langchain_openai import ChatOpenAI

# model = ChatOpenAI()

model = ChatOpenAI(    
    base_url="http://localhost:12434/engines/v1",
    api_key="docker", 
    temperature=0, 
    model = "ai/smollm2:360M-Q4_K_M")

In [ ]:
from pydantic import BaseModel, Field


class EvaluationSchema(BaseModel):
    feedback: str = Field(descrption="Detailed feedback for the essay")
    score: int = Field(description="Score out of 10", ge=0, le=10)

In [ ]:
structured_model = model.with_structured_output(EvaluationSchema)

In [ ]:
essay = """Artificial Intelligence (AI) is rapidly reshaping India’s economy and society. It is being applied in agriculture to improve crop yields through predictive analytics and smart irrigation systems. In healthcare, AI supports early disease detection, telemedicine, and efficient hospital management. The education sector benefits from AI-driven personalized learning platforms that make quality education more accessible. In manufacturing, robotics and machine learning enhance productivity and reduce costs. AI also strengthens financial services, enabling fraud detection, automated customer support, and inclusive digital banking.

India’s government has launched initiatives like NITI Aayog’s National Strategy for AI and the AI for India 2030 program to promote innovation and adoption. These efforts aim to position India as a global leader in the digital economy, projected to reach $1 trillion in value. However, challenges remain, including data privacy concerns, skill shortages, and infrastructure gaps. Ethical issues such as job displacement and the digital divide must also be addressed to ensure inclusive growth.

If harnessed responsibly, AI could add 1.3 percentage points to India’s annual GDP growth and create new opportunities across industries. Thus, AI is not just a technological tool but a catalyst for India’s ambition to become a $5 trillion economy by 2027."""

In [ ]:
prompt = f"Evaluate the language quality of an essay and provide feedback and assign a score out of 10\n {essay}"

structured_model.invoke(prompt)


In [ ]:
# Define state
class AgentState(TypedDict):
    essay: str
    language_feedback: str
    analysis_feedback: str
    clarity_feedback: str
    overall_feedback: str
    individual_scores: Annotated[list[int], operator.add]
    avg_score: float

In [ ]:
def evaluate_language(state: AgentState):
    """Evaluate language"""
    essay = state["essay"]
    prompt = f"Evaluate the language quality of an essay and provide feedback and assign a score out of 10\n {essay}"
    output: EvaluationSchema = structured_model.invoke(prompt)
    return {"language_feedback": output.feedback, "individual_scores": [output.score]}

In [ ]:
def evaluate_analysis(state: AgentState):
    """Evaluate analysis"""
    essay = state["essay"]
    prompt = f"Evaluate the depth of analysis of an essay and provide feedback and assign a score out of 10\n {essay}"
    output: EvaluationSchema = structured_model.invoke(prompt)
    return {"analysis_feedback": output.feedback, "individual_scores": [output.score]} 

In [ ]:
def evaluate_thought(state: AgentState):
    """Evaluate based on clarity of thought"""
    essay = state["essay"]
    prompt = f"Evaluate based on clarity of thoughts of an essay and provide feedback and assign a score out of 10\n {essay}"
    output: EvaluationSchema = structured_model.invoke(prompt)
    return {"clarity_feedback": output.feedback, "individual_scores": [output.score]} 

In [ ]:
def final_evaluation(state: AgentState):
    """Final evaluation"""
    essay = state["essay"]
    language_feedback = state["language_feedback"]
    analysis_feedback = state["analysis_feedback"]
    clarity_feedback = state["clarity_feedback"]
    individual_scores = state["individual_scores"]
    
    prompt = f"Based on the following feedbacks care a summaurized feedback\n ##1. language feedback: {language_feedback}\n ##2. depth of analysis feedback: {analysis_feedback}\n ##3. clarity of thought feedback {clarity_feedback}\n"
    overall_feedback = model.invoke(prompt).content

    avg_score = sum(individual_scores)/len(individual_scores)

    return {
        "overall_feedback": overall_feedback,
        "avg_score": avg_score
    }
    

In [ ]:
# Create a graph

graph = StateGraph(AgentState)

# Add node
graph.add_node("evaluate_language", evaluate_language)
graph.add_node("evaluate_analysis", evaluate_analysis)
graph.add_node("evaluate_thought", evaluate_thought)
graph.add_node("final_evaluation", final_evaluation)

# add edges
graph.add_edge(START, "evaluate_language")
graph.add_edge(START, "evaluate_analysis")
graph.add_edge(START, "evaluate_thought")

graph.add_edge("evaluate_language", "final_evaluation")
graph.add_edge("evaluate_analysis", "final_evaluation")
graph.add_edge("evaluate_thought", "final_evaluation")

graph.add_edge("final_evaluation", END)

# compile the graph
workflow = graph.compile()

In [ ]:
# visualize graph
from IPython.display import Image
Image(workflow.get_graph().draw_mermaid_png())

In [ ]:
initial_state = {
    "essay": essay,
    "language_feedback": "",
    "analysis_feedback": "",
    "clarity_feedback": "",
    "overall_feedback": "",
    "individual_scores": [],
    "avg_score": 0
}

final_state = workflow.invoke(initial_state)

In [ ]:
print(final_state)